# Analyze durable episodes

This product-neutral notebook reads `run_episode` artifacts through the installed `collusion_experiment` package. Canonical metrics and plots live in `src/collusion_experiment/`. Episodes are the treatment-level units; stage and round views are descriptive. This notebook makes no model calls and never rewrites source run artifacts.


## Package and analysis imports

Install the repository package in this notebook runtime (for example `python -m pip install -e ".[analysis]"` locally, or an uploaded wheel in Colab) before running. No repository URL or environment-specific scientific path is assumed.


In [ ]:
from pathlib import Path
import csv
import collusion_experiment
from collusion_experiment.config import STRUCTURED_CONFIG
SPECIFICATION_VERSION = STRUCTURED_CONFIG.specification_version  # Explicit V2 analysis namespace.
from collusion_experiment.metrics import (
    ArtifactConsistencyError, discover_run_directories, episode_summary,
    load_episode_artifacts, paired_prior_communication_difference,
    stage_summaries, treatment_descriptives,
)
from collusion_experiment.public_signal_inference import public_signal_episode_tests
from collusion_experiment.plotting import (
    plot_episode_participation_timeline, plot_public_signal_mapping_timeline,
    plot_treatment_metric_distribution, plot_paired_replicate_differences,
)
print("Package:", collusion_experiment.__file__)


## Select source runs

From the repository root or `notebooks/`, the default is `results/<specification_id>/runs/`. Otherwise the default is relative to the current working directory. Set `RESULTS_ROOT` to another mounted or local directory if needed. Drive mounting is optional and external to the scientific analysis.


In [ ]:
cwd = Path.cwd()
if (cwd / "pyproject.toml").is_file() and (cwd / "src" / "collusion_experiment").is_dir():
    REPO_ROOT = cwd
elif (cwd.parent / "pyproject.toml").is_file() and (cwd.parent / "src" / "collusion_experiment").is_dir():
    REPO_ROOT = cwd.parent
else:
    REPO_ROOT = None
RESULTS_ROOT = (REPO_ROOT or cwd) / "results" / SPECIFICATION_VERSION / "runs"
print("Source results root:", RESULTS_ROOT.resolve())


## Load runs and inspect status

Corrupt runs are reported in the status list; they are excluded from calculations. Failed and incomplete runs remain visible but receive no full-episode outcome metrics.


In [ ]:
run_dirs = discover_run_directories([RESULTS_ROOT]) if RESULTS_ROOT.is_dir() else []
episodes = []
invalid_runs = []
for run_dir in run_dirs:
    try:
        episodes.append(load_episode_artifacts(run_dir))
    except ArtifactConsistencyError as exc:
        invalid_runs.append({"run_dir": str(run_dir), "status": "invalid/corrupt", "error": str(exc)})
status_rows = [
    {"run_dir": str(ep.run_dir), "status": ep.status,
     "last_completed_round": ep.last_completed_round,
     "failure_type": (ep.failure or {}).get("failure", {}).get("exception_type")}
    for ep in episodes
] + invalid_runs
print("Run status counts:", {status: sum(row["status"] == status for row in status_rows) for status in sorted({row["status"] for row in status_rows})})
print("Invalid runs:", invalid_runs if invalid_runs else "none")


## Episode, stage, and treatment descriptions

Stage rows describe within-episode paths. Treatment statistics aggregate completed episodes first; rounds are never pooled as independent replicates.


In [ ]:
episode_rows = [episode_summary(ep) for ep in episodes]
stage_rows = [row for ep in episodes for row in stage_summaries(ep)]
treatment_rows = treatment_descriptives(episode_rows)
print("Episode summaries:", len(episode_rows))
print("Stage summaries:", len(stage_rows))
print("Treatment descriptions:", treatment_rows)


In [ ]:
PAIRED_SIGNAL_CONDITION = "public"
PAIRED_METRIC = "joint_profit"
paired_rows = paired_prior_communication_difference(
    episode_rows, signal_condition=PAIRED_SIGNAL_CONDITION, metric=PAIRED_METRIC,
)
print("Paired replicate differences:", paired_rows if paired_rows else "No matching replicates")


## Exploratory public-signal episode test

This cell reads the completed JSONL run artifacts loaded above. In each public-signal episode, the statistic is the larger of the two counts matching opposite signal-to-agent assignments. Under the strong null that changing fair, independent signal draws would leave bids unchanged, the count for either fixed assignment is Binomial(sole-bid rounds, 0.5). The exact p-value accounts for choosing the better assignment. Holm adjustment covers all 40 public-signal episodes in blocks 1-20. This analysis was added after some results were inspected and is exploratory.


In [ ]:
PUBLIC_SIGNAL_BLOCKS = range(1, 21)  # Completed sprint cohort; two public episodes per block.
public_signal_rows = (
    public_signal_episode_tests(episodes, block_ids=PUBLIC_SIGNAL_BLOCKS) if episodes else []
)
print("Public-signal episode tests:", len(public_signal_rows))
for row in public_signal_rows:
    print(
        f"block {row['block']:02d} | earlier chat {row['earlier_communication']} | "
        f"S={row['mapping_score']:.4f} | m={row['sole_bid_rounds']} | "
        f"k={row['best_matches']} | exact p={row['exact_p']:.6g} | "
        f"Holm p={row['holm_adjusted_p']:.6g}"
    )
print("Holm p < 0.05:", [
    (row["block"], row["earlier_communication"])
    for row in public_signal_rows if row["holm_adjusted_p"] < 0.05
])


## Descriptive figures

Figures are returned by package functions. Choose a completed episode for public mapping; no figure is written unless the export cell is enabled.


In [ ]:
figures = {}
completed = [ep for ep in episodes if ep.status == "completed"]
if completed:
    figures["participation"] = plot_episode_participation_timeline(completed[0])[0]
    figures["treatment_joint_profit"] = plot_treatment_metric_distribution(
        episode_rows, metric="joint_profit"
    )[0]
    public_completed = next((ep for ep in completed if ep.config["signal_condition"] == "public"), None)
    if public_completed:
        figures["public_mapping"] = plot_public_signal_mapping_timeline(public_completed)[0]
    if any(row["status"] == "paired" for row in paired_rows):
        figures["paired_joint_profit"] = plot_paired_replicate_differences(
            paired_rows, metric=PAIRED_METRIC
        )[0]
print("Figures prepared:", list(figures))


## Optional derived exports

Exports are off by default and use a separate analysis directory. The source run directories remain read-only.


In [ ]:
EXPORT_DERIVED = False
ANALYSIS_OUTPUT_ROOT = (REPO_ROOT or cwd) / "results" / SPECIFICATION_VERSION / "analysis"
if EXPORT_DERIVED:
    output_root = ANALYSIS_OUTPUT_ROOT.resolve()
    if any(output_root.is_relative_to(run_dir.resolve()) for run_dir in run_dirs):
        raise ValueError("Analysis exports must be outside scientific run directories.")
    output_root.mkdir(parents=True, exist_ok=True)
    for filename, rows in (
        ("episode_summary.csv", episode_rows),
        ("stage_summary.csv", stage_rows),
        ("paired_differences.csv", paired_rows),
        ("public_signal_episode_tests.csv", public_signal_rows),
    ):
        if rows:
            target = output_root / filename
            with target.open("x", encoding="utf-8", newline="") as handle:
                writer = csv.DictWriter(handle, fieldnames=list(dict.fromkeys(k for row in rows for k in row)))
                writer.writeheader()
                writer.writerows(rows)
            print("Wrote:", target)
    for name, fig in figures.items():
        target = output_root / f"{name}.png"
        with target.open("xb") as handle:
            fig.savefig(handle, format="png", dpi=300, bbox_inches="tight")
        print("Wrote:", target)
